<a href="https://colab.research.google.com/github/olga-terekhova/pdf-utilities/blob/main/ShiftPDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shift PDFs

## How to use

To **shift** PDF pages:  
1) Prepare the PDF file you want to process. Upload it to the file storage, one file only.  
2) In the [Set parameters](#scrollTo=vjip4SJ4hzz1&line=2&uniqifier=1) below set the name for the output pdf. E.g. *output.pdf*.
3) In the [Set parameters](#scrollTo=vjip4SJ4hzz1&line=2&uniqifier=1) set offset X (horizontal) and offset Y (vertical) in inches. Positive values for shifts to the right and to the bottom, negative values for shifts to the left and to the bottom. E.g. *0.5*, *-1.2*.  
4) Run all cells in the notebook (Runtime - Run all or Ctrl-F9).  
5) Download the output pdf from the Files area (Refresh to see the newly created copied file).  

In [1]:
# @title Set parameters

shifted_pdf_path = 'output.pdf' # @param {type:"string"}
offset_x_in = -2            # @param {type:"number"}
offset_y_in = -1                 # @param {type:"number"}

print(shifted_pdf_path)


output.pdf


## Code (you can collapse this section)

### Install, import, initialize  

In [2]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 73.5 MB/s eta 0:00:00


In [3]:
import os
import pymupdf

### Shift the PDF file

In [4]:
def get_file():
  """
  Get the first PDF file in the current directory.
  Return the file name and a message.
  """
  pdf_files = []
  for filename in os.listdir():
      if filename.endswith('.pdf'):
          pdf_files.append(filename)

  if shifted_pdf_path in pdf_files:
    return "", "File " + shifted_pdf_path + " already exists. No action taken. Do you want to delete PDF files first?"

  if len(pdf_files) == 0:
    return "", "No PDF files found. No action taken."

  # sort pdf_files in the alphabetical order
  pdf_files.sort()

  # take the first PDF file
  pdf_file = pdf_files[0]
  print(pdf_file)

  return pdf_file, "OK"

In [5]:
def shift_pdf(output_path, offset_x_in, offset_y_in):
    """
    Shift the contents of all pages in a PDF by (offset_x_in, offset_y_in) inches
    and save the result to a new PDF.

    Args:
        output_path (str): Path to the output PDF.
        offset_x_in (float): Horizontal offset in inches (positive = right).
        offset_y_in (float): Vertical offset in inches (positive = bottom).
    """
    # --- Convert inches to PDF points ---
    POINTS_PER_INCH = 72.0
    offset_x = offset_x_in * POINTS_PER_INCH
    offset_y = offset_y_in * POINTS_PER_INCH

    # Get the input PDF file (a first PDF file found in the root directory)
    input_pdf, input_pdf_response = get_file()
    if input_pdf == "":  # no file to process
        return input_pdf_response

    # Open source and destination PDFs
    src_doc = pymupdf.open(input_pdf)
    dst_doc = pymupdf.open()

    for page_number in range(src_doc.page_count):
        # Load a page and get its rectangle
        src_page = src_doc.load_page(page_number)
        rect = src_page.rect

        # Create a new blank page with the same size
        dst_page = dst_doc.new_page(width=rect.width, height=rect.height)

        # Define a new rect shifted by the offsets
        shifted_rect = pymupdf.Rect(
            rect.x0 + offset_x,
            rect.y0 + offset_y,
            rect.x1 + offset_x,
            rect.y1 + offset_y
        )

        # Draw the source page into the shifted rectangle
        dst_page.show_pdf_page(shifted_rect, src_doc, page_number)

    # Save the shifted PDF
    dst_doc.save(output_path)
    dst_doc.close()
    src_doc.close()

    result =f"✅ Saved shifted PDF to: {output_path}" + \
     f"Applied offset: {offset_x_in:.2f} in (x), {offset_y_in:.2f} in (y) → " + \
       f"{offset_x:.1f} pt, {offset_y:.1f} pt"
    return result


In [6]:
result = shift_pdf(shifted_pdf_path, offset_x_in, offset_y_in)

kenguru_2025_class_1-2.pdf


## Result

In [7]:
print(result)

✅ Saved shifted PDF to: output.pdfApplied offset: -2.00 in (x), -1.00 in (y) → -144.0 pt, -72.0 pt
